In [0]:
-- ============================================
-- Gold: daily_kpis
-- ============================================

USE CATALOG shoply_analytics;
USE SCHEMA gold;

CREATE OR REPLACE VIEW daily_kpis AS
WITH
-- Sessions aggregated by day
sessions_daily AS (
    SELECT
        DATE(session_start) AS date,
        COUNT(*) AS sessions,
        COUNT(DISTINCT user_id) AS unique_users
    FROM shoply_analytics.silver.sessions
    GROUP BY DATE(session_start)
),

-- Page views aggregated by day
pageviews_daily AS (
    SELECT
        DATE(timestamp) AS date,
        COUNT(*) AS page_views
    FROM shoply_analytics.silver.page_views
    GROUP BY DATE(timestamp)
),

-- Conversions aggregated by day
conversions_daily AS (
    SELECT
        DATE(timestamp) AS date,
        COUNT(*) AS conversions
    FROM shoply_analytics.silver.conversion
    GROUP BY DATE(timestamp)
),

-- Campaign attributed sessions
campaign_sessions AS (
    SELECT
        DATE(session_start) AS date,
        COUNT(*) AS campaign_sessions
    FROM shoply_analytics.silver.sessions
    WHERE campaign_id IS NOT NULL
    GROUP BY DATE(session_start)
),

-- Campaign attributed conversions
campaign_conversions AS (
    SELECT
        DATE(timestamp) AS date,
        COUNT(*) AS campaign_conversion
    FROM shoply_analytics.silver.conversion
    WHERE campaign_id IS NOT NULL
    GROUP BY DATE(timestamp)
)

-- Final KPI table
SELECT
    COALESCE(s.date, p.date, c.date) AS date,
    s.sessions,
    s.unique_users,
    p.page_views,
    c.conversions,
    cs.campaign_sessions,
    cc.campaign_conversion,
    -- Derived metrics
    CASE WHEN s.sessions > 0 THEN p.page_views / s.sessions ELSE NULL END AS avg_pages_per_session,
    CASE WHEN s.sessions > 0 THEN c.conversions / s.sessions ELSE NULL END AS conversion_rate
FROM sessions_daily s
FULL OUTER JOIN pageviews_daily p ON s.date = p.date
FULL OUTER JOIN conversions_daily c ON COALESCE(s.date, p.date) = c.date
LEFT JOIN campaign_sessions cs ON COALESCE(s.date, p.date, c.date) = cs.date
LEFT JOIN campaign_conversions cc ON COALESCE(s.date, p.date, c.date) = cc.date;
